In [6]:
import kfp
from google.cloud import aiplatform
from google_cloud_pipeline_components._implementation.model import GetVertexModelOp
from google_cloud_pipeline_components.v1.endpoint.create_endpoint.component import \
    endpoint_create as EndpointCreateOp
from google_cloud_pipeline_components.v1.endpoint.deploy_model.component import \
    model_deploy as ModelDeployOp

In [7]:
PROJECT_ID = "project-c8d59959-80e8-496f-96a"
GCP_REGION = "us-central1"
DISPLAY_NAME = "train-eval-deploy-26Aug26-1"
MACHINE_TYPE = "n1-standard-4"
BUCKET_URI = "gs://custom-trin-bucket"
AUTOML_MODEL = "projects/90514214177/locations/us-central1/models/1809286515679821824"
CUSTOM_MODEL = "projects/90514214177/locations/us-central1/models/5429248230225346560"
CUSTOM_MODEL_RAW = "gs://custom-trin-bucket/custom_model_test1.keras"
VERTEX_DATASET = "projects/90514214177/locations/us-central1/datasets/7580367688300494848"
TEST_DATA_URI = "gs://bucket-for-kpf-project/test_UCI_Credit_Card.csv"

In [8]:
@kfp.dsl.component(
    packages_to_install=["google-cloud-aiplatform"],
    base_image="python:3.12",
)
def eval_automl_model(
    project: str,
    region: str,
    automl_model: str,
) -> float:
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)

    auto_ml_model = aiplatform.Model(automl_model)
    automl_evaluations = auto_ml_model.list_model_evaluations()
    
    for evaluation in automl_evaluations:
        evaluation = evaluation.to_dict()
        metrics = evaluation["metrics"]
        automl_acc = metrics["auPrc"]
            
    return automl_acc

@kfp.dsl.component(
    packages_to_install=["google-cloud-aiplatform", "tensorflow",
                         "pandas", "fsspec", "gcsfs", "scikit-learn"],
    base_image="python:3.12",
)
def eval_custom_model(
    project: str,
    region: str,
    custom_model_raw: str,
    test_data_uri: str,
) -> float:
    from google.cloud import aiplatform
    import tensorflow as tf
    import pandas as pd
    from sklearn.preprocessing import StandardScaler

    aiplatform.init(project=project, location=region)

    custom_model = tf.keras.models.load_model(custom_model_raw)

    df = pd.read_csv(test_data_uri)
    test_labels = df.pop('default.payment.next.month')
    scaler = StandardScaler()
    df = scaler.fit_transform(df)

    loss, custom_acc = custom_model.evaluate(df, test_labels, verbose=0)
    custom_acc = custom_acc
            
    return custom_acc

@kfp.dsl.component
def print_comp(text: str):
    print(text)

    

/var/tmp/ipykernel_3577/3919136099.py:53: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @kfp.dsl.component


In [9]:
@kfp.dsl.pipeline(
    name=DISPLAY_NAME,
    pipeline_root=BUCKET_URI,
    )
def pipeline(
    project: str = PROJECT_ID,
    region: str = GCP_REGION,
    automl_model: str = AUTOML_MODEL,
    custom_model_raw: str = CUSTOM_MODEL_RAW,
    test_data_uri: str = TEST_DATA_URI,
    custom_model: str = CUSTOM_MODEL,
):

    automl_model_acc = eval_automl_model(project=project, region=region, automl_model=automl_model)

    custom_model_acc = eval_custom_model(project=project, region=region,
                                         custom_model_raw=custom_model_raw,
                                        test_data_uri=test_data_uri)

    print_comp(
        text = f"AutmoML acc: {automl_model_acc.output}, Custmol model acc: {custom_model_acc.output}")

    endpoint_op = EndpointCreateOp(
        project=PROJECT_ID,
        location=GCP_REGION,
        display_name=DISPLAY_NAME,
    )
    
    with kfp.dsl.If(automl_model_acc.output > custom_model_acc.output):
        model_task = GetVertexModelOp(
            model_name=automl_model,
        )
        ModelDeployOp(
            model=model_task.outputs["model"],
            endpoint=endpoint_op.outputs["endpoint"],
            dedicated_resources_min_replica_count=1,
            dedicated_resources_max_replica_count=1,
            dedicated_resources_machine_type=MACHINE_TYPE,
        )
    with kfp.dsl.Else():
        model_task = GetVertexModelOp(
            model_name=custom_model,
        )
        ModelDeployOp(
            model=model_task.outputs["model"],
            endpoint=endpoint_op.outputs["endpoint"],
            dedicated_resources_min_replica_count=1,
            dedicated_resources_max_replica_count=1,
            dedicated_resources_machine_type=MACHINE_TYPE,
        )

    

In [10]:
kfp.compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='eval_pipeline_26Aug_v2.yaml'
)
!gsutil cp -r eval_pipeline_26Aug_v2.yaml gs://custom-trin-bucket/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://eval_pipeline_26Aug_v2.yaml [Content-Type=application/yaml]...
/ [1 files][ 44.3 KiB/ 44.3 KiB]                                                
Operation completed over 1 objects/44.3 KiB.                                     


In [11]:
aiplatform.init(
    project=PROJECT_ID,
    location=GCP_REGION,
)

# Prepare the pipeline job
job = aiplatform.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path="eval_pipeline_26Aug_v2.yaml",
    pipeline_root=BUCKET_URI,
    enable_caching=True,
)

job.submit()

Creating PipelineJob
PipelineJob created. Resource name: projects/90514214177/locations/us-central1/pipelineJobs/train-eval-deploy-26aug26-1-20260826141334
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/90514214177/locations/us-central1/pipelineJobs/train-eval-deploy-26aug26-1-20260826141334')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/train-eval-deploy-26aug26-1-20260826141334?project=90514214177
